In [1]:
import os

from tqdm import tqdm
from ase.db import connect

from fairchem.core.datasets import AseDBDataset

W0409 14:30:07.588000 42043 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# Validation Dataset

## Load Dataset

In [2]:
#!time curl -O "https://dl.fbaipublicfiles.com/opencatalystproject/data/omol/250514/val.tar.gz"
#!tar -xzvf val.tar.gz

In [2]:
dataset_path = "/Volumes/JAC_Backup/OpenFF/Meta-OMol25/val"
output_path = os.path.split(dataset_path)[0]
dataset = AseDBDataset({"src": dataset_path})

## Filter Atoms in ASE Databases

In [3]:
#target_ds_ids = ["elytes", "biomolecules", "metal_complexes", "spice", "orbnet_denali", "geom_orca6", "reactivity", "ani2x", "rgd", "trans1x"]
target_ds_ids = ["metal_complexes"]

os.makedirs(os.path.join(output_path,"ase_databases"), exist_ok=True)
ase_dbs = {ds_name: connect(os.path.join(output_path, f"ase_databases/{ds_name}.db")) for ds_name in target_ds_ids}

In [4]:
for id in tqdm(dataset.ids):
    atoms = dataset.get_atoms(id)
    if atoms.info["data_id"] not in target_ds_ids:
        continue
    atoms.info["omol25-index"] = id
    atoms.info["omol25-split"] = "val"
    ase_dbs[f"{atoms.info['data_id']}"].write(atoms, data=atoms.info)


100%|██████████| 2762021/2762021 [1:35:03<00:00, 484.27it/s] 


In [6]:
# Get atoms.info later with:
#    last_rowid = ase_dbs["geom_orca6"].count()   
#    row = ase_dbs["geom_orca6"].get(id=last_rowid)
#    atoms2 = row.toatoms()
#    atoms2.info = row.data

# Training Dataset

In [7]:
#!time curl -O "https://dl.fbaipublicfiles.com/opencatalystproject/data/omol/250514/train.tar.gz"
#!tar -xzvf train.tar.gz

In [5]:
dataset_path_train = "/Volumes/JAC_Backup/OpenFF/Meta-OMol25/train"
output_path = os.path.split(dataset_path)[0]
dataset_train = AseDBDataset({"src": dataset_path_train})

## Filter Atoms in ASE Databases

In [7]:
#target_ds_ids = ["spice", "orbnet_denali", "geom_orca6", "ani2x"]
target_ds_ids = ["metal_complexes"]

os.makedirs(os.path.join(output_path,"ase_databases"), exist_ok=True)
ase_dbs = {ds_name: connect(os.path.join(output_path, f"ase_databases/{ds_name}.db")) for ds_name in target_ds_ids}

In [8]:
for id in tqdm(dataset_train.ids):
    atoms = dataset_train.get_atoms(id)
    if atoms.info["data_id"] not in target_ds_ids:
        continue
    atoms.info["omol25-index"] = id
    atoms.info["omol25-split"] = "train"
    ase_dbs[f"{atoms.info['data_id']}"].write(atoms, data=atoms.info)

  7%|▋         | 7517304/101666280 [38:22:47<480:40:50, 54.41it/s]     


OperationalError: disk I/O error

## Check ASE Databases

In [6]:
target_ds_ids = ["spice"]

output_path = "/Volumes/JAC_Backup/OpenFF/Meta-OMol25/"
os.makedirs(os.path.join(output_path,"ase_databases"), exist_ok=True)
ase_dbs = {ds_name: connect(os.path.join(output_path, f"ase_databases/{ds_name}.db")) for ds_name in target_ds_ids}

In [12]:
row = ase_dbs["spice"].get(id=1)
print(row)
for k, v in row.data.items():
    print(k, v)  # stored metadata

<AtomsRow: formula=C15H32N8O5P, keys=>
source spice/SPICE_Amino_Acid_Ligand_v1_0_spice_19N_HIS_1_1_1/orca.tar.zst
reference_source None
data_id spice
charge 1
spin 1
num_atoms 61
num_electrons 232
num_ecp_electrons 0
n_scf_steps 14
n_basis 1382
unrestricted False
nl_energy 23.92289411058149
integrated_densities [116.00001113 116.00001113 232.00002226]
homo_energy [-9.96981646]
homo_lumo_gap [7.92496243]
s_squared 0.0
s_squared_dev 0.0
warnings ['B97M-V or wB97M-V functional requested.libXC variant has now been activated.Any manually given DFX/HFX/RangeSep scaling parameters are IGNORED!', 'Analytic gradients for DFT-NL dispersion can only be done using the self-consistent method,Switching to SCNL', 'Self-consistent DFT-NL dispersion not compatible with AutoTRAH.Switching AutoTRAH off']
mulliken_charges [-0.700886 -0.652469 -0.844639 -0.722071 -0.52343  -0.522757  0.786505
 -0.054489  0.551825 -0.790385  0.307361 -0.353675 -0.469128  0.220812
 -0.116891 -0.202675 -0.578955 -0.015619  0.

In [ ]:
from datasets import load_from_disk, DatasetDict

spice_metadata_path = "/Volumes/JAC_Backup/OpenFF/Meta-OMol25/huggingface_datasets/spice_metadata"
spice_metadata = load_from_disk(spice_metadata_path)

if isinstance(spice_metadata, DatasetDict):
    split_name = next(iter(spice_metadata.keys()))
    print(f"First row from split '{split_name}':")
    print(spice_metadata[split_name][0])
else:
    print(spice_metadata[0])

In [11]:
print(row.data.keys())

dict_keys(['source', 'reference_source', 'data_id', 'charge', 'spin', 'num_atoms', 'num_electrons', 'num_ecp_electrons', 'n_scf_steps', 'n_basis', 'unrestricted', 'nl_energy', 'integrated_densities', 'homo_energy', 'homo_lumo_gap', 's_squared', 's_squared_dev', 'warnings', 'mulliken_charges', 'lowdin_charges', 'nbo_charges', 'composition', 'omol25-index', 'omol25-split'])
